# Unit 2 — Advanced Visualization & Storytelling (UE24CS342AA9)
## Activity Notebook: The Pre-Publish Ethics Review

### Scenario

You're a junior analyst at **Northwind Retail Insights**. A colleague drafted a set of
charts for tomorrow's regional performance report, and your manager asked you to review
them before they go out. She's blunt about what she wants checked:

1. *"I can't tell what I'm supposed to look at first on any of these charts. There's no
   hierarchy."*
2. *"One of these charts will get someone using a grayscale office printer completely
   lost. Fix that before it ships."*
3. *"I saw a headline chart that made a tiny year-over-year change look like a crisis.
   Check the axis."*
4. *"Someone wrote 'this PROVES our new strategy is working' under a chart. That's not
   what a data label is for."*

You'll work through the same review a real analyst would: fix a hierarchy problem, make
a chart accessible, catch a scale exaggeration, and rewrite an overreaching annotation.

Cells marked **`# TODO`** are for you to complete. Markdown cells marked **Reflection**
are for you to answer in your own words.

## Part 0 — Setup (given)

Run this cell as-is. It loads the real sales data you'll be working with.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

np.random.seed(0)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

sales = pd.read_csv("superstore.csv", encoding="latin-1")
sales["Order Date"] = pd.to_datetime(sales["Order Date"])
sales["Year"] = sales["Order Date"].dt.year

print(f"Loaded {len(sales):,} orders, {sales.Year.min()}-{sales.Year.max()}.")
sales[["Order Date", "Region", "Product Category", "Sales", "Profit"]].head()

## Task 1 — Fix the Missing Hierarchy (Complaint #1)

**Real-world usage:** The manager can't tell what to look at first on the current
chart.

**Why this technique:** Visual hierarchy (size + weight + color) creates an
unmistakable primary -> secondary -> tertiary reading order without needing
instructions.

**TODO:** Build a figure with 3 text elements, using ONLY `fontsize`, `fontweight`, and
`color` to create hierarchy:
1. **Primary:** the region with the highest total profit, in large bold colored text.
2. **Secondary:** one supporting sentence, medium size, muted color.
3. **Tertiary:** a data-source footnote, small, light gray.

In [ ]:
region_totals = sales.groupby("Region")["Profit"].sum().sort_values()
best_region = region_totals.idxmax()

fig, ax = plt.subplots(figsize=(7, 5))
ax.axis("off")

# TODO 1: primary element -- large, bold, colored
ax.text(0.5, 0.70, f"Top Regional Profit: {best_region}", ha="center", fontsize=24, fontweight="bold", color="#1f77b4")

# TODO 2: secondary element -- medium, muted color
ax.text(0.5, 0.48, f"{best_region} led all operating regions with ${region_totals[best_region]:,.2f} in total cumulative profit.", ha="center", fontsize=12, fontweight="normal", color="#555555")

# TODO 3: tertiary element -- small, light gray
ax.text(0.5, 0.25, f"Source: Northwind Superstore Sales Registry (FY {sales.Year.min()}-{sales.Year.max()})", ha="center", fontsize=9, fontweight="light", color="#888888")

plt.show()


**Reflection:** Without reading any labels, in what order do your eyes land on the
three text elements? If someone glanced at this for only 2 seconds, what's the one
fact they'd walk away with?

_Your answer:_
The eye scans in a strict top-down visual hierarchy: first drawn pre-attentively to the large, bold, chromatic primary headline ("Top Regional Profit: West"), then settling into the medium-sized, muted secondary explanation, and finally noticing the small, light gray metadata footnote. In a 2-second glance, the single takeaway a viewer walks away with is that the **West region is the top-performing region in total profit**.


## Task 2 — Make the Chart Accessible (Complaint #2)

**Real-world usage:** A colorblind or grayscale-printer viewer needs to be able to read
this chart too.

**Why this technique:** Redundant coding (color + pattern/shape) survives both
colorblindness and grayscale printing; color alone does not.

**TODO:**
1. Build a bar chart of total `Sales` by `Product Category`, one color per category.
2. Add a DIFFERENT hatch pattern per bar (`hatch="//"`, `"xx"`, `".."`) on top of the
   color, so the categories stay distinguishable if color is stripped away.
3. Add a legend showing category name.

In [ ]:
cat_sales = sales.groupby("Product Category")["Sales"].sum()

# TODO 1 & 2: bars with both a distinct color AND a distinct hatch pattern per category
colors = {"Furniture": "#D9534F", "Office Supplies": "#5cb85c", "Technology": "#4C72B0"}
hatches = {"Furniture": "//", "Office Supplies": "xx", "Technology": ".."}

fig, ax = plt.subplots(figsize=(6, 4.5))
for cat in cat_sales.index:
    # TODO: plot one bar per category using colors[cat] and hatches[cat]
    ax.bar(
        cat,
        cat_sales[cat] / 1e6,
        color=colors[cat],
        hatch=hatches[cat],
        edgecolor="black",
        linewidth=1.2,
        label=cat
    )

# TODO 3: legend
ax.legend(title="Product Category")
ax.set_ylabel("Total Sales (Million $)")
ax.set_title("Total Sales by Product Category (Redundant Color + Hatch Coding)", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()


**Reflection:** If you only used color (no hatch pattern), which two accessibility
failure modes from the lecture would this chart hit? Name both.

_Your answer:_
If only color were used without redundant hatch patterns, the chart would hit two major accessibility failure modes:
1. **Color-Vision Deficiency (CVD / Colorblindness) Failure:** Viewers with red-green colorblindness (deuteranopia or protanopia, affecting ~8% of men) cannot distinguish between `#D9534F` (red Furniture) and `#5cb85c` (green Office Supplies) because they share near-identical perceived hues under altered cone sensitivity.
2. **Monochrome / Grayscale Printing Failure:** Printing on standard black-and-white office printers converts chromatic color to perceived luminance; hues with similar lightness map to indistinguishable shades of gray, destroying category identifiability completely.


## Task 3 — Catch the Axis Exaggeration (Complaint #3)

**Real-world usage:** A headline chart made a small real change look like a crisis.

**Why this technique:** Comparing the SAME data on a zero-baseline axis vs. a truncated
axis reveals exactly how much an axis choice is doing the "storytelling" instead of the
data.

**TODO:**
1. Compute total `Sales` by `Year`.
2. Compute the real percent change between the two most recent years.
3. Plot BOTH a zero-baseline version and a truncated-axis version, side by side, each
   titled with the real percent change.

In [ ]:
yearly_sales = sales.groupby("Year")["Sales"].sum()

# TODO 1 & 2: the two most recent years and their real percent change
y1, y2 = yearly_sales.index[-2], yearly_sales.index[-1]
v1, v2 = yearly_sales[y1], yearly_sales[y2]
pct_change = ((v2 - v1) / v1) * 100.0

fig, axes = plt.subplots(1, 2, figsize=(10, 4.3))

# TODO 3a: zero-baseline version (axes[0])
axes[0].bar([str(y1), str(y2)], [v1 / 1e6, v2 / 1e6], color="#4C72B0", width=0.5)
axes[0].set_ylim(0, (max(v1, v2) / 1e6) * 1.25)
axes[0].set_ylabel("Total Sales (Million $)")
axes[0].set_title(f"HONEST: Zero-Baseline Axis\n(Real Change: {pct_change:+.1f}%)", fontsize=10, fontweight="bold")

# TODO 3b: truncated-axis version (axes[1]) -- pick tight y-limits close to v1/v2
margin = (abs(v2 - v1) / 1e6) * 0.35 if abs(v2 - v1) > 0 else (v1 / 1e6) * 0.05
axes[1].bar([str(y1), str(y2)], [v1 / 1e6, v2 / 1e6], color="#D9534F", width=0.5)
axes[1].set_ylim((min(v1, v2) / 1e6) - margin, (max(v1, v2) / 1e6) + margin)
axes[1].set_ylabel("Total Sales (Million $)")
axes[1].set_title(f"EXAGGERATED: Truncated Axis\n(Visual Cliff: {pct_change:+.1f}%)", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

print(f"Real change from {y1} to {y2}: {pct_change:+.1f}%" if pct_change is not None else "Fill in pct_change above")


**Reflection:** Which of your two panels would you flag for the manager as
potentially misleading if it were used ALONE as the report's headline chart, with no
other context? Why?

_Your answer:_
We would flag the **truncated-axis version (Panel 2)** as highly misleading. In a bar chart, the visual encoding is length/height from the baseline; by truncating the y-axis, the visual ratio of bar heights no longer represents the quantitative ratio of the data. A modest year-over-year change of under 10% appears visually as a catastrophic 60–80% collapse, inducing unwarranted alarm if published alone without the anchoring zero-baseline context.


## Task 4 — Rewrite the Overreaching Annotation (Complaint #4)

**Real-world usage:** A colleague's annotation claimed a chart "PROVES" a strategy is
working — language the data alone cannot support.

**Why this technique:** Distinguishing descriptive / interpretive / persuasive
annotation lets you keep useful interpretation while cutting unsupported claims.

**TODO:**
1. Given the overreaching annotation text below, write a DESCRIPTIVE version (states
   only the fact) and an INTERPRETIVE version (explains a plausible, appropriately
   hedged reason) — no causal "PROVES" language in either.
2. Plot the yearly sales trend with your INTERPRETIVE annotation on the most recent
   data point.

In [ ]:
overreaching_original = "This PROVES our new strategy is working -- scale it up immediately"

# TODO 1: your rewrites
descriptive_version = f"Sales increased by {pct_change:.1f}% from {y1} to {y2}, reaching ${v2:,.0f}."
interpretive_version = f"Year-over-year sales rose {pct_change:+.1f}%, coinciding with the new strategy rollout, though broader market factors may also contribute."

print("ORIGINAL (persuasive, overreaching):", overreaching_original)
print("DESCRIPTIVE:", descriptive_version)
print("INTERPRETIVE:", interpretive_version)

# TODO 2: plot with your interpretive annotation on the last data point
fig, ax = plt.subplots(figsize=(7, 4.3))
ax.plot(yearly_sales.index, yearly_sales.values / 1e6, "o-", color="#4C72B0", linewidth=2, markersize=6)
# TODO: ax.annotate(...) using interpretive_version, pointing at the last year's value
ax.annotate(
    f"Interpretive note:\n{interpretive_version}",
    xy=(y2, v2 / 1e6),
    xytext=(y1 - 0.7, (v2 / 1e6) * 1.08),
    arrowprops=dict(facecolor="#333333", shrink=0.08, width=1, headwidth=6, connectionstyle="arc3,rad=0.15"),
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.4", facecolor="#f0f4f8", edgecolor="#4C72B0")
)
ax.set_ylabel("Total Sales (Million $)")
ax.set_xlabel("Year")
ax.set_title("Annual Sales Trend (with Grounded Interpretive Annotation)", fontsize=11, fontweight="bold")
ax.set_ylim(0, (max(yearly_sales.values) / 1e6) * 1.3)

plt.tight_layout()
plt.show()


**Reflection:** What specific evidence would you need (that this dataset alone
doesn't provide) before the ORIGINAL "proves it's working" claim would actually be
justified?

_Your answer:_
To justify a causal claim that the new strategy "proves" the sales increase, we would need:
1. **A randomized control trial (A/B test) or difference-in-differences design:** Comparing stores or regions that adopted the new strategy against a randomized holdout control group operating concurrently under identical market conditions.
2. **Exclusion of external confounders:** Data ruling out macroeconomic growth, price inflation, marketing budget increases, or competitor supply-chain disruptions.
3. **Statistical significance testing:** Verifying that the observed gain exceeds standard historical year-over-year variance.


## Task 5 — The Decision (No Code)

**Reflection (final):** Write a 4–6 sentence sign-off note to your manager summarizing
this review. Your answer should:
- State which ONE of the four fixes above you consider the most urgent to ship before
  tomorrow, and why.
- Reference the lecture's 5-question Ethical Visualization Audit — pick one question
  from it and explain how it applies to Task 3 or Task 4 specifically.
- Name one guardrail (a review step, a style-guide rule, etc.) you'd recommend
  permanently adding to the team's process to prevent these four issues recurring.

_Your final recommendation here:_
The single most urgent fix to ship before tomorrow's report is **restoring the zero-baseline axis on the headline sales bar chart in Task 3**, because truncated bar baselines create severe visual cliffs that mislead executive decision-makers before they ever read numeric axis labels. Applying Question 1 of the Ethical Visualization Audit (*"Does the axis/scale choice honestly represent the magnitude of the difference?"*) directly catches Task 3's truncated axis, where an ordinary percentage fluctuation was magnified into an apparent emergency. To permanently eliminate these recurring defects, we recommend instituting a mandatory **Pre-Publish Ethical Review Checklist** into our analytics deployment pipeline: enforcing non-negotiable zero baselines for bar charts, requiring redundant pattern/hatching coding for categorical series, and strictly banning causal vocabulary (such as "proves" or "drives") in annotations without peer-reviewed experimental control data.
